In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 5.5
fig_height = 3.5
fig_format = 'pdf'
fig_dpi = 300
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'RTpcR2l0SHViXEJpb0ltYWdpbmdBSVxkb2Nz'
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"C:\\Users\\leer\\.conda\\envs\\BioImagingAI_ch10\\Lib\\importlib\\_bootstrap.py": 1785932743.0, "C:\\Users\\leer\\.conda\\envs\\BioImagingAI_ch10\\Lib\\importlib\\_bootstrap_external.py": 1785932743.0, "C:\\Users\\leer\\.conda\\envs\\BioImagingAI_ch10\\Lib\\zipimport.py": 1785932743.0, "C:\\Users\\leer\\.conda\\envs\\BioImagingAI_ch10\\Lib\\codecs.py": 1785932743.0, "C:\\Users\\leer\\.conda\\envs\\BioImagingAI_ch10\\Lib\\encodings\\aliases.py": 1785932743.0, "C:\\Users\\leer\\.conda\\envs\\BioImagingAI_ch10\\Lib\\encodings\\__init__.py": 1785932743.0, "C:\\Users\\leer\\.conda\\envs\\BioImagingAI_ch10\\Lib\\encodings\\utf_8.py": 1785932743.0, "C:\\Users\\leer\\.conda\\envs\\BioImagingAI_ch10\\Lib\\encodings\\cp1252.py": 1785932743.0, "C:\\Users\\leer\\.conda\\envs\\BioImagingAI_ch10\\Lib\\abc.py": 1785932743.0, "C:\\Users\\leer\\.conda\\envs\\BioImagingAI_ch10\\Lib\\io.py": 1785932743.0, "C:\\Users\\leer\\.conda\\envs\\BioImagingAI_ch10\\Lib\\stat.py": 1785932743.0, "C:\\Users\\leer\\

In [2]:
#| echo: false

import skimage as ski
import numpy as np
from skimage.draw import ellipse as draw_ellipse
import matplotlib.pyplot as plt
from skimage.filters import threshold_otsu, gaussian
from skimage.measure import label, regionprops
from skimage.morphology import closing, footprint_rectangle, erosion, dilation, disk, remove_small_objects
from skimage.color import label2rgb
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from scipy.ndimage import binary_fill_holes, shift as ndimage_shift, distance_transform_edt


def make_ground_truth(image, sigma=2, min_distance=10, max_size=500):
    """Segment DAPI-stained nuclei using Gaussian smoothing + Otsu + watershed.

    Returns
    -------
    bw : ndarray  binary foreground mask (after smoothing, thresholding, size filter)
    filled : ndarray  binary mask of watershed regions (same as bw for solid nuclei)
    label_image : ndarray  integer label image, one label per nucleus
    """
    smoothed = gaussian(image, sigma=sigma)
    bw = closing(smoothed > threshold_otsu(smoothed), footprint_rectangle((3, 3)))
    bw = remove_small_objects(bw, max_size=max_size)
    dist = distance_transform_edt(bw)
    peaks = peak_local_max(dist, min_distance=min_distance, labels=bw)
    markers = np.zeros_like(bw, dtype=np.int32)
    markers[tuple(peaks.T)] = np.arange(1, len(peaks) + 1)
    label_image = watershed(-dist, markers, mask=bw)
    return bw, (label_image > 0), label_image


def make_undersegmented(filled, erode_radius=8, jitter=6, seed=7):
    """Erode each cell mask inward and nudge it by a small random offset."""
    eroded = label(erosion(filled, disk(erode_radius)))
    rng = np.random.default_rng(seed)
    out = np.zeros_like(eroded)
    for lbl in np.unique(eroded)[1:]:
        mask = (eroded == lbl).astype(float)
        dy, dx = rng.integers(-jitter, jitter + 1, size=2)
        shifted = ndimage_shift(mask, (dy, dx), order=0)
        out[shifted > 0] = lbl
    return out


def make_oversegmented(filled, max_dilate_radius=5, seed=1):
    """Dilate each cell by a different random amount (0 to max_dilate_radius)."""
    cell_labels = label(filled)
    rng = np.random.default_rng(seed)
    out = np.zeros_like(cell_labels)
    for lbl in np.unique(cell_labels)[1:]:  # skip background
        mask = cell_labels == lbl
        radius = int(rng.integers(0, max_dilate_radius + 1))
        dilated = dilation(mask, disk(radius)) if radius > 0 else mask
        out[dilated] = lbl
    return out


def make_missing_cells(label_image, interior_fraction=1/6, seed=42):
    """Remove all border cells and a random fraction of interior cells."""
    h, w = label_image.shape
    out = label_image.copy()
    props = regionprops(label_image)
    border_lbl = {p.label for p in props
                  if p.bbox[0] == 0 or p.bbox[1] == 0
                  or p.bbox[2] == h or p.bbox[3] == w}
    interior_lbl = [p.label for p in props if p.label not in border_lbl]
    rng = np.random.default_rng(seed)
    drop = rng.choice(interior_lbl,
                      size=max(1, int(len(interior_lbl) * interior_fraction)),
                      replace=False)
    for lbl in border_lbl | set(drop):
        out[out == lbl] = 0
    return out


def make_holey(bw, erode_radius=2, fp_radius=18, seed=42):
    """Thin the membrane rings so the central holes become larger,
    and seed a few false-positive detections in the background."""
    out = label(erosion(bw, disk(erode_radius)))
    h, w = out.shape
    rng = np.random.default_rng(seed)
    next_lbl = out.max() + 1
    positions = [(int(h * 0.88), int(w * 0.08)),   # bottom left corner
                 (int(h * 0.25), int(w * 0.08))]    # 75% up the left side
    for cy, cx in positions:
        r_rad = int(rng.uniform(fp_radius * 0.8, fp_radius * 1.2))
        c_rad = int(rng.uniform(fp_radius * 0.5, fp_radius * 0.9))
        rotation = rng.uniform(0, np.pi)
        rr, cc = draw_ellipse(cy, cx, r_rad, c_rad, shape=(h, w), rotation=rotation)
        fp = np.zeros((h, w), dtype=bool)
        fp[rr, cc] = True
        out[fp & (out == 0)] = next_lbl
        next_lbl += 1
    return out

In [3]:
#| code-fold: true
#| label: fig-image
#| fig-cap: Our cells, with the ground truth overlaid. This ground truth is the output of a Cellpose[@stringer2021] model, but in practice it should be manually annotated.

import contextlib, io, logging
from cellpose import models

img = ski.data.cells3d()[21:]
image = img[18][1]  # nucleus channel

# Run Cellpose quietly for display
logging.getLogger('cellpose').setLevel(logging.ERROR)
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    _model = models.CellposeModel(gpu=False, model_type='nuclei')
    _cp_labels, _, _ = _model.eval(image, diameter=None, channels=[0, 0])

# Use make_ground_truth for bw/filled/label_image so that failure modes
# (holes, split nuclei) are visible in the downstream figures
bw, filled, label_image = make_ground_truth(image)

image_label_overlay = label2rgb(_cp_labels, image=image, bg_label=0, alpha=0.2)
plt.imshow(image_label_overlay)
plt.axis("off")
plt.show()

<Figure size 1650x1050 with 1 Axes>

In [4]:
#| code-fold: true
#| label: fig-segmentations
#| fig-cap: Four examples of segmentations from different models. Each has a distinct failure mode that is hard to spot at a glance.
#| fig-subcap:
#|   - 'Undersegmented: masks shrink away from cell borders'
#|   - 'Oversegmented: masks are too large, adjacent cells merge'
#|   - 'False negatives: missing cells and split nuclei.'
#|   - 'Holes and false positives: gaps inside cells and spurious detections.'
#| layout-ncol: 2

predictions = [
    make_undersegmented(filled),
    make_oversegmented(filled),
    make_missing_cells(label_image),
    make_holey(bw),
]

for pred in predictions:
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(label2rgb(pred, image=image, bg_label=0))
    ax.axis("off")
    plt.show()

<Figure size 1200x1200 with 1 Axes>

<Figure size 1200x1200 with 1 Axes>

<Figure size 1200x1200 with 1 Axes>

<Figure size 1200x1200 with 1 Axes>

In [5]:
#| fig-align: center
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch, Polygon as MplPolygon
import numpy as np

navy, orange, green = "#1a35e0", "#e47a10", "#2a9d2a"

fig, ax = plt.subplots(figsize=(5, 3))
ax.set_xlim(0, 5)
ax.set_ylim(0, 5)
ax.set_xticks(np.arange(0, 6, 0.5))
ax.set_yticks(np.arange(0, 6, 0.5))
ax.grid(True, color="lightgray", lw=0.8)
ax.tick_params(labelbottom=False, labelleft=False, length=0)

A = (0.5, 1, 2, 4)      # x, y, width, height
B = (1.5, 1.5, 3, 3)

ax.add_patch(Rectangle(A[:2], A[2], A[3], facecolor=navy, alpha=0.5))
ax.add_patch(Rectangle(B[:2], B[2], B[3], facecolor=orange, alpha=0.5))

# Union: outline that follows the actual shape of A ∪ B
union_verts = [
    (A[0],      A[1]),           # bottom-left of A
    (A[0]+A[2], A[1]),           # bottom-right of A
    (A[0]+A[2], B[1]),           # down to bottom of B
    (B[0]+B[2], B[1]),           # bottom-right of B
    (B[0]+B[2], B[1]+B[3]),      # top-right of B
    (A[0]+A[2], B[1]+B[3]),      # back to A's right edge at top of B
    (A[0]+A[2], A[1]+A[3]),      # top-right of A
    (A[0],      A[1]+A[3]),      # top-left of A
]
ax.add_patch(MplPolygon(union_verts, closed=True,
                        facecolor="none", edgecolor="black", lw=2, linestyle="--"))

ix0, iy0 = max(A[0], B[0]), max(A[1], B[1])
ix1, iy1 = min(A[0]+A[2], B[0]+B[2]), min(A[1]+A[3], B[1]+B[3])
ax.add_patch(Rectangle((ix0, iy0), ix1-ix0, iy1-iy0,
                        facecolor=green, alpha=0.85, edgecolor="white", hatch="////"))

legend_elements = [
    Patch(facecolor=navy, alpha=0.5, label="A"),
    Patch(facecolor=orange, alpha=0.5, label="B"),
    Patch(facecolor=green, alpha=0.85, hatch="////", label="A ∩ B"),
    Patch(facecolor="none", edgecolor="black", lw=2, linestyle="--", label="A ∪ B"),
]
ax.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, -0.02),
          ncol=2, frameon=False, fontsize=14)

<Figure size 1500x900 with 1 Axes>

In [6]:
#| fig-align: center
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
import numpy as np

navy, orange, green = "#1a35e0", "#e47a10", "#2a9d2a"

H, W = 300, 500  # raster grid

def ellipse_mask(cx, cy, rx, ry):
    """Boolean mask for an ellipse in axis coordinates [0,10] x [0,6]."""
    x = np.linspace(0, 10, W)
    y = np.linspace(0, 6,  H)
    xx, yy = np.meshgrid(x, y)
    return ((xx - cx) / rx)**2 + ((yy - cy) / ry)**2 <= 1

# Label image: 0=bg, 1=GT only (orange), 2=pred only (blue), 3=overlap (green)
img = np.zeros((H, W), dtype=int)

# True positives: GT and prediction ellipses slightly offset
tps = [  # gt_x, gt_y, gt_rx, gt_ry,  dx, dy  (prediction offset)
    (2.0, 3.0, 1.0, 0.8,   0.4,  0.3),
    (5.0, 3.8, 0.9, 1.0,  -0.3,  0.4),
    (7.5, 2.0, 1.0, 0.85,  0.35,-0.3),
    (4.5, 1.2, 0.8, 0.7,  -0.25, 0.3),
]
for cx, cy, rx, ry, dx, dy in tps:
    gt   = ellipse_mask(cx,    cy,    rx,   ry)
    pred = ellipse_mask(cx+dx, cy+dy, rx,   ry)
    img[gt   & ~pred] = 1
    img[pred & ~gt  ] = 2
    img[gt   &  pred] = 3

# False negatives — ground truth only (orange)
for cx, cy, rx, ry in [(3.0, 5.0, 0.85, 0.7), (8.5, 4.5, 0.9, 0.8)]:
    img[ellipse_mask(cx, cy, rx, ry)] = 1

# False positives — prediction only (blue)
for cx, cy, rx, ry in [(1.0, 1.0, 0.8, 0.9), (6.5, 5.0, 0.9, 0.75)]:
    img[ellipse_mask(cx, cy, rx, ry)] = 2

cmap = ListedColormap(["#ffffff", orange, navy, green])
fig, ax = plt.subplots(figsize=(5, 3))
ax.imshow(img, cmap=cmap, vmin=0, vmax=3,
          extent=[0, 10, 0, 6], origin="lower", aspect="auto")
ax.axis("off")

legend_elements = [
    Patch(facecolor=green,  label="Matched (TP)"),
    Patch(facecolor=orange, label="Ground truth only (FN)"),
    Patch(facecolor=navy,   label="Prediction only (FP)"),
]
ax.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, -0.02),
          ncol=1, frameon=False, fontsize=9)

<Figure size 1500x900 with 1 Axes>

In [7]:
#| fig-align: center
import matplotlib.pyplot as plt
import numpy as np

navy, orange, green = "#1a35e0", "#e47a10", "#2a9d2a"

fig, ax = plt.subplots(figsize=(5, 3))
ax.set_xlim(0, 5)
ax.set_ylim(0, 5)
ax.set_xticks(np.arange(0, 6, 0.5))
ax.set_yticks(np.arange(0, 6, 0.5))
ax.grid(True, color="lightgray", lw=0.8)
ax.tick_params(labelbottom=False, labelleft=False, length=0)

theta_f = np.linspace(0, 2 * np.pi, 300)
theta_c = np.linspace(0, 2 * np.pi, 20, endpoint=False)

# A: ground truth boundary (circle)
A_line = np.stack([2.5 + 1.5 * np.cos(theta_f), 2.5 + 1.5 * np.sin(theta_f)], axis=1)
A_pts  = np.stack([2.5 + 1.5 * np.cos(theta_c), 2.5 + 1.5 * np.sin(theta_c)], axis=1)

# B: predicted boundary — slightly larger and offset
B_line = np.stack([2.9 + 1.8 * np.cos(theta_f), 2.7 + 1.8 * np.sin(theta_f)], axis=1)
B_pts  = np.stack([2.9 + 1.8 * np.cos(theta_c), 2.7 + 1.8 * np.sin(theta_c)], axis=1)

ax.plot(A_line[:, 0], A_line[:, 1], color=navy, lw=2, label="A")
ax.plot(B_line[:, 0], B_line[:, 1], color=orange, lw=2, label="B")

# Draw nearest-neighbour lines from each point in A to its closest point in B
dists = np.linalg.norm(A_pts[:, None] - B_pts[None], axis=2)
nn = np.argmin(dists, axis=1)
for i in range(len(A_pts)):
    ax.plot([A_pts[i, 0], B_pts[nn[i], 0]], [A_pts[i, 1], B_pts[nn[i], 1]],
            color="lightgray", lw=0.7, zorder=1)

# Highlight the worst-case pair (the Hausdorff point)
w = np.argmax(dists[np.arange(len(A_pts)), nn])
ax.annotate("", xy=(B_pts[nn[w], 0], B_pts[nn[w], 1]),
            xytext=(A_pts[w, 0], A_pts[w, 1]),
            arrowprops=dict(arrowstyle="<->", color=green, lw=2.0), zorder=3)
mid = (A_pts[w] + B_pts[nn[w]]) / 2
ax.text(mid[0] - 0.1, mid[1] - 0.3, "$d_H$", color=green, fontsize=13, va="center")

ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02),
          ncol=2, frameon=False, fontsize=14)

<Figure size 1500x900 with 1 Axes>

In [8]:
#| fig-align: center
import matplotlib.pyplot as plt
import numpy as np

navy, orange = "#1a35e0", "#e47a10"

# Synthetic precision-recall curve
recall = np.array([0.0, 0.15, 0.35, 0.55, 0.70, 0.82, 0.92, 1.0])
prec   = np.array([1.0, 0.92, 0.82, 0.70, 0.58, 0.44, 0.28, 0.0])

fig, ax = plt.subplots(figsize=(5, 3))
ax.set_xlim(-0.05, 1.1)
ax.set_ylim(-0.05, 1.15)
ax.set_xlabel("Recall ($R$)", fontsize=11)
ax.set_ylabel("Precision ($P$)", fontsize=11)
ax.set_xticks([0.0, 0.5, 1.0])
ax.set_yticks([0.0, 0.5, 1.0])
ax.grid(True, color="lightgray", lw=0.8)

# Step function and shaded area under the curve = AP
ax.step(recall, prec, where="post", color=navy, lw=2)
ax.fill_between(recall, prec, step="post", alpha=0.25, color=navy, label="AP")

# Mark the individual (R_k, P_k) threshold points
ax.scatter(recall[1:-1], prec[1:-1], color=orange, s=45, zorder=5,
           label="$(R_k, P_k)$")

ax.legend(loc="upper right", frameon=False, fontsize=11)

<Figure size 1500x900 with 1 Axes>

In [9]:
#| label: fig-tracking-graph
#| fig-cap: |-
#|   Cell tracking represented as a graph. (Top) Segmentation masks
#|   detect a few cells in each of three time points; the blue cell persists
#|   across all three frames while the orange cell divides into two daughters
#|   (green and purple) at $t = 2$. (Bottom) The corresponding tracking graph places one
#|   node per detected cell, with edges linking the same cell across frames.
#|   Node columns and colors align with the masks above, and the division
#|   appears as a node with two outgoing edges.
#| echo: false

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle

# Palette consistent with the rest of the chapter
BLUE = "#1f77b4"
ORANGE = "#ff7f0e"
GREEN = "#2ca02c"
PURPLE = "#9467bd"

LABEL_FONT_SIZE = 9
TITLE_FONT_SIZE = 10
NODE_SIZE = 220
EDGE_WIDTH = 2
N_FRAMES = 3

# Scene: a few cells tracked across three time points.
#   Cell A (blue):   present in all 3 frames, drifting horizontally but holding
#                    a fixed vertical position so its lineage is a straight line.
#   Cell B (orange): present in frames 0 and 1, divides at frame 2 into two
#                    daughters (green and purple) -> introduces a branch.
# Each occurrence stores its position (x, y) inside a unit frame; the y value
# is reused as the vertical position of the matching graph node so that each
# blob lines up with its node below.
CELLS = {
    "A0": dict(frame=0, pos=(0.30, 0.70), color=BLUE,   seed=1, r=(0.15, 0.10)),
    "A1": dict(frame=1, pos=(0.42, 0.70), color=BLUE,   seed=2, r=(0.15, 0.10)),
    "A2": dict(frame=2, pos=(0.52, 0.70), color=BLUE,   seed=3, r=(0.15, 0.10)),
    "B0": dict(frame=0, pos=(0.62, 0.30), color=ORANGE, seed=4, r=(0.16, 0.11)),
    "B1": dict(frame=1, pos=(0.58, 0.32), color=ORANGE, seed=5, r=(0.11, 0.08)),
    "C2": dict(frame=2, pos=(0.40, 0.22), color=GREEN,  seed=6, r=(0.13, 0.09)),
    "D2": dict(frame=2, pos=(0.70, 0.40), color=PURPLE, seed=7, r=(0.13, 0.09)),
}
EDGES = [
    ("A0", "A1"), ("A1", "A2"),
    ("B0", "B1"),
    ("B1", "C2"), ("B1", "D2"),  # division
]


def blob_outline(cx, cy, rx, ry, seed):
    """x, y arrays for a nucleus-like outline: an ellipse with a slight,
    low-frequency wobble so it reads as an organic shape rather than a circle."""
    theta = np.linspace(0, 2 * np.pi, 80)
    rng = np.random.default_rng(seed)
    a, b = rng.uniform(-1, 1, 2)
    r = 1 + 0.04 * np.sin(2 * theta + 2 * a) + 0.025 * np.cos(3 * theta + 3 * b)
    return cx + rx * r * np.cos(theta), cy + ry * r * np.sin(theta)


def draw_blob(ax, cell):
    cx, cy = cell["pos"]
    rx, ry = cell["r"]
    x, y = blob_outline(cx, cy, rx, ry, cell["seed"])
    base = mcolors.to_rgb(cell["color"])
    fill = tuple(0.55 * c + 0.45 for c in base)  # lightened fill
    ax.fill(x, y, color=fill, zorder=1)
    ax.plot(x, y, color=cell["color"], lw=1.8, zorder=2)


fig = plt.figure(figsize=(5,5*(5/7)))
gs = fig.add_gridspec(2, N_FRAMES, height_ratios=[1.0, 1.15],
                      hspace=0.28, wspace=0.12)

# Top row: segmentation masks at each time point
for t in range(N_FRAMES):
    ax = fig.add_subplot(gs[0, t])
    ax.set_facecolor("white")
    for cell in CELLS.values():
        if cell["frame"] == t:
            draw_blob(ax, cell)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"$t = {t}$", fontsize=TITLE_FONT_SIZE)
    # solid black frame of even weight on all four sides
    for s in ax.spines.values():
        s.set_visible(True)
        s.set_color("black")
        s.set_linewidth(1.2)

# Bottom row: the tracking graph spanning all frames
gax = fig.add_subplot(gs[1, :])
pos = {cid: (c["frame"], c["pos"][1]) for cid, c in CELLS.items()}
# edges drawn as solid lines colored by the child node, matching the other
# lineage figures in this chapter
for u, v in EDGES:
    (xu, yu), (xv, yv) = pos[u], pos[v]
    gax.plot([xu, xv], [yu, yv], color=CELLS[v]["color"], lw=EDGE_WIDTH,
             zorder=1)
for cid, (x, y) in pos.items():
    gax.scatter([x], [y], s=NODE_SIZE, color=CELLS[cid]["color"],
                edgecolors="white", linewidths=1.5, zorder=2)
for t in range(N_FRAMES):
    gax.plot([t, t], [0, 1.0], color="0.85", lw=1.0, zorder=0)
    gax.text(t, 1.04, f"$t = {t}$", ha="center", va="bottom",
             fontsize=LABEL_FONT_SIZE, color="0.4")
gax.set_xlim(-0.4, N_FRAMES - 1 + 0.4)
gax.set_ylim(0, 1.14)
gax.set_xlabel("Time", fontsize=LABEL_FONT_SIZE)
gax.set_ylabel("Tracking graph", fontsize=LABEL_FONT_SIZE)
gax.set_xticks([])
gax.set_yticks([])
for s in gax.spines.values():
    s.set_visible(False)

fig.subplots_adjust(left=0.05, right=0.97, top=0.92, bottom=0.06)

<Figure size 1500x1071.43 with 4 Axes>

In [10]:
#| label: fig-track-errors
#| fig-cap: |-
#|   Common types of tracking errors. (a) This example prediction demonstrates multiple types of 
#|   node and edge errors: false negative node, false negative edges and false positive node. (b) This example 
#|   shows a missed division as the result of a missed edge between a parent and a daughter. (c) This example 
#|   shows a division where the parent was correctly identified, but one of the daughters was incorrectly assigned 
#|   to a different node. (d) This is an example of an identity switch where two nodes are swapped and assigned to 
#|   the incorrect lineage part way through the lineage.
#| echo: false

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle

# Palette mimicking the reference figure
RED = "#d62728"
GREEN = "#2ca02c"
BLUE = "#1f77b4"
PURPLE = "#9467bd"
ORANGE = "#ff7f0e"
GRAY = "0.6"

# How much darker the predicted tracks are relative to ground truth (1.0 = same)
PRED_SHADE = 0.6

LABEL_FONT_SIZE = 8
TITLE_FONT_SIZE = 10
NODE_SIZE = 50
EDGE_WIDTH = 2


def darken(color, factor=PRED_SHADE):
    """Return a darker shade of `color` by scaling toward black in RGB."""
    r, g, b = mcolors.to_rgb(color)
    return (r * factor, g * factor, b * factor)


def plot_tree(ax, nodes, edges, missing_edges=None, x_offset=0.0,
              shade=1.0):
    """Draw one lineage tree.

    nodes:         {id: (x, frame, color)}
    edges:         [(u, v), ...]  drawn as solid lines, colored by child node
    missing_edges: [(u, v), ...]  drawn as faint dashed lines (false negatives)
    shade:         scale node/edge colors toward black (use < 1 for Pred trees)
    """
    pos = {n: (x + x_offset, f) for n, (x, f, _) in nodes.items()}

    def shaded(c):
        return darken(c, shade) if shade != 1.0 else c

    for u, v in (missing_edges or []):
        (xu, yu), (xv, yv) = pos[u], pos[v]
        ax.plot([xu, xv], [yu, yv], color=GRAY, lw=edge_width,
                ls=(0, (2, 2)), zorder=1)

    for u, v in edges:
        (xu, yu), (xv, yv) = pos[u], pos[v]
        ax.plot([xu, xv], [yu, yv], color=shaded(nodes[v][2]), lw=EDGE_WIDTH,
                zorder=1)

    for n, (x, f, color) in nodes.items():
        ax.scatter([x + x_offset], [f], s=NODE_SIZE, color=shaded(color),
                   zorder=2, edgecolors="none")
    return pos


# --------------------------------------------------------------------------
# Panel builders. Each returns (true_nodes, true_edges, pred_nodes,
# pred_edges, pred_missing) and draws its own annotations.
# Convention: frame increases downward (y axis inverted at the end).
# --------------------------------------------------------------------------

def panel_node_edge(ax, gap):
    """FP node + FN node/edge on a single track."""
    title = "Node & edge\nerrors"
    # True: straight track, frames 0-4
    tn = {i: (0, i, BLUE) for i in range(5)}
    te = [(i, i + 1) for i in range(4)]
    plot_tree(ax, tn, te)

    # Pred: frame-2 node missing (FN gap), an extra detection at frame 0 (FP)
    pn = {i: (0, i, BLUE) for i in [0, 1, 3, 4]}
    pn["fp"] = (0.7, 0, ORANGE)
    pe = [(0, 1), (3, 4)]
    plot_tree(ax, pn, pe, x_offset=gap, shade=PRED_SHADE)
    return title, tn, pn


def panel_missed_division(ax, gap):
    """Division present in True, missing in Pred (FN edge)."""
    title = "Missed\ndivision"
    # True: red parent 0-2, divides into green + blue at frame 3
    tn = {
        0: (0.5, 0, ORANGE), 1: (0.5, 1, ORANGE), 2: (0.5, 2, ORANGE),
        3: (0.0, 3, GREEN), 4: (0.0, 4, GREEN),
        5: (1.0, 3, BLUE), 6: (1.0, 4, BLUE),
    }
    te = [(0, 1), (1, 2), (2, 3), (3, 4), (2, 5), (5, 6)]
    plot_tree(ax, tn, te)

    # Pred: the division is missed, so the parent links linearly into one
    # daughter (no branch). Nodes keep their correct (true) lineage colors, so
    # the spurious parent->green link shows as a color change along the track;
    # the disconnected blue daughter is orphaned.
    pn = {
        0: (0.5, 0, ORANGE), 1: (0.5, 1, ORANGE), 2: (0.5, 2, ORANGE),
        3: (0.0, 3, GREEN), 4: (0.0, 4, GREEN),
        5: (1.0, 3, BLUE), 6: (1.0, 4, BLUE),
    }
    pe = [(0, 1), (1, 2), (2, 3), (3, 4), (5, 6)]
    plot_tree(ax, pn, pe, x_offset=gap, shade=PRED_SHADE)
    return title, tn, pn


def panel_wrong_daughter(ax, gap):
    """Correct parent, one correct daughter, one incorrect daughter (I)."""
    title = "Incorrect\ndaughter"
    tn = {
        0: (0.5, 0, ORANGE), 1: (0.5, 1, ORANGE), 2: (0.5, 2, ORANGE),
        3: (0.0, 3, GREEN), 4: (0.0, 4, GREEN),
        5: (1.0, 3, BLUE), 6: (1.0, 4, BLUE),
    }
    te = [(0, 1), (1, 2), (2, 3), (3, 4), (2, 5), (5, 6)]
    plot_tree(ax, tn, te)

    # PRED: parent divides into the correct green daughter and a wrong cell
    # (orange); the true blue daughter still exists but is now an orphan track
    pn = {
        0: (0.5, 0, ORANGE), 1: (0.5, 1, ORANGE), 2: (0.5, 2, ORANGE),
        3: (0.0, 3, GREEN), 4: (0.0, 4, GREEN),
        "w1": (1.0, 3, ORANGE), "w2": (1.0, 4, ORANGE),
        5: (1.8, 3, BLUE), 6: (1.8, 4, BLUE),
    }
    pe = [(0, 1), (1, 2), (2, 3), (3, 4), (2, "w1"), ("w1", "w2"), (5, 6)]
    plot_tree(ax, pn, pe, x_offset=gap, shade=PRED_SHADE)
    return title, tn, pn


def panel_identity_switch(ax, gap):
    """Two lineages whose identities swap partway through (I)."""
    title = "Identity\nswitch"
    # True: two parallel tracks
    tn = {
        ("a", i): (0.0, i, BLUE) for i in range(5)
    }
    tn.update({("b", i): (1.0, i, ORANGE) for i in range(5)})
    te = [(("a", i), ("a", i + 1)) for i in range(4)]
    te += [(("b", i), ("b", i + 1)) for i in range(4)]
    plot_tree(ax, tn, te)

    # Pred: nodes keep their correct (true) lineage colors; the identity switch
    # is shown by the linking edges crossing columns at frame 2->3, so each
    # predicted track follows the wrong cell after the swap.
    pn = {
        ("a", 0): (0.0, 0, BLUE), ("a", 1): (0.0, 1, BLUE),
        ("a", 2): (0.0, 2, BLUE),
        ("a", 3): (0.0, 3, BLUE), ("a", 4): (0.0, 4, BLUE),
        ("b", 0): (1.0, 0, ORANGE), ("b", 1): (1.0, 1, ORANGE),
        ("b", 2): (1.0, 2, ORANGE),
        ("b", 3): (1.0, 3, ORANGE), ("b", 4): (1.0, 4, ORANGE),
    }
    pe = [(("a", 0), ("a", 1)), (("a", 1), ("a", 2)),
          (("b", 0), ("b", 1)), (("b", 1), ("b", 2)),
          (("a", 3), ("a", 4)), (("b", 3), ("b", 4)),
          # crossing links
          (("a", 2), ("b", 3)), (("b", 2), ("a", 3))]
    plot_tree(ax, pn, pe, x_offset=gap, shade=PRED_SHADE)
    return title, tn, pn

def plot_figure():
    panels = [
        panel_node_edge,
        panel_missed_division,
        panel_wrong_daughter,
        panel_identity_switch,
    ]
    gap = 3.6  # horizontal offset of the Pred tree from the True tree

    # fig, axes = plt.subplots(1, len(panels), figsize=(5, 5*(2.8/7)))
    fig, axes = plt.subplots(2, 2, figsize=(5, 5))
    labels = ["a)", "b)", "c)", "d)"]
    pad_x, label_y = 1.0, -1.05  # box padding and True/Pred label height
    box_top, box_bot = -1.3, 4.6  # vertical box extent (frames run 0..4)
    for ax, builder, lab in zip(axes.ravel(), panels, labels):
        title, tn, pn = builder(ax, gap)
        true_xs = [x for x, _, _ in tn.values()]
        pred_xs = [x + gap for x, _, _ in pn.values()]

        # center each label over the horizontal extent of its lineage
        ax.text((min(true_xs) + max(true_xs)) / 2, label_y, "Ground\nTruth",
                ha="center", va="top", linespacing=0.9,
                fontsize=LABEL_FONT_SIZE)
        ax.text((min(pred_xs) + max(pred_xs)) / 2, label_y, "Predicted",
                ha="center", va="top", fontsize=LABEL_FONT_SIZE)

        # square box hugging the panel content
        xmin = min(true_xs + pred_xs) - pad_x
        xmax = max(true_xs + pred_xs) + pad_x
        ax.add_patch(Rectangle(
            (xmin, box_top), xmax - xmin, box_bot - box_top,
            fill=False, edgecolor="black", linewidth=1.0))

        ax.set_title(f"{lab} {title}", fontsize=TITLE_FONT_SIZE,
                     fontweight="bold", pad=6)
        ax.set_ylim(box_bot + 0.15, box_top - 0.15)  # inverted y
        ax.set_xlim(xmin - 0.1, xmax + 0.1)
        ax.axis("off")

    fig.tight_layout(w_pad=0.4)

plot_figure()

<Figure size 1500x1500 with 4 Axes>

In [11]:
#| code-fold: true
#| code-summary: Show code to generate a set of lineage trees with regular divisions
import networkx as nx

def build_lineage_tree(
    n_starting_nodes: int = 10,
    n_frames: int = 30,
    division_every: int = 5,
) -> nx.DiGraph:
    """
    Build a lineage tree as a directed graph.

    - Starts with `n_starting_nodes` roots at frame 0
    - Every `division_every` frames, each live cell divides into 2 daughters
    - Between division frames, cells persist as a single descendant

    Nodes are labeled like:
        cell_id = "c0", "c1", ...
    and store:
        - frame
        - root
    """
    G = nx.DiGraph()

    cell_counter = 0
    live_cells = []

    # Create starting nodes at frame 0
    for root in range(n_starting_nodes):
        node = cell_counter
        cell_counter += 1
        G.add_node(node, frame=0, root=root)
        live_cells.append(node)

    # Advance through frames
    for frame in range(1, n_frames + 1):
        next_live_cells = []
        is_division_frame = (frame % division_every == 0)

        for parent in live_cells:
            root = G.nodes[parent]["root"]

            if is_division_frame:
                # Parent divides into two daughters
                for _ in range(2):
                    child = cell_counter
                    cell_counter += 1
                    G.add_node(child, frame=frame, root=root)
                    G.add_edge(parent, child)
                    next_live_cells.append(child)
            else:
                # Parent continues as one descendant
                child = cell_counter
                cell_counter += 1
                G.add_node(child, frame=frame, root=root)
                G.add_edge(parent, child)
                next_live_cells.append(child)

        live_cells = next_live_cells

    return G


def sparsify_edges(g, mod=3):
    del_edges = []
    for i, edge in enumerate(g.edges()):
        if i % mod == 0:
            del_edges.append(edge)
    g.remove_edges_from(del_edges)


def remove_divs(g):
    div_edges = []
    for node in g.nodes:
        if g.out_degree(node) == 2:
            div_edges.extend(g.edges(node))

    g.remove_edges_from(div_edges)


def remove_nodes(g, mod=4):
    del_nodes = []
    for i, node in enumerate(sorted(g.nodes)):
        if g.nodes[node]['frame'] > 0 and i % mod == 0:
            del_nodes.append(node)
    g.remove_nodes_from(del_nodes)


gt = build_lineage_tree(n_starting_nodes=10, n_frames=30, division_every=5)

In [12]:
#| echo: false

# Plotting utilities

def annotate_positions(g, spacing=1.0, offset=0.0):
    # Assign x coordinates to each node for plotting.
    # Last-frame daughter leaves are spaced by `spacing`.
    if len(g.nodes) == 0:
        return

    last_frame_leaves = [n for n in g.nodes if g.out_degree(n) == 0]

    last_frame_leaves.sort()
    for i, node in enumerate(last_frame_leaves):
        g.nodes[node]['x'] = offset + i * spacing

    for node in sorted(g.nodes, key=lambda n: g.nodes[n]['frame'], reverse=True):
        if 'x' in g.nodes[node]:
            continue
        children = list(g.successors(node))
        if children:
            g.nodes[node]['x'] = sum(g.nodes[c]['x'] for c in children) / len(children)
        else:
            g.nodes[node]['x'] = offset


def get_single_tree(g, root=0, max_frame=None):
    del_nodes = []
    for n, attrs in g.nodes.items():
        if attrs['root'] != root:
            del_nodes.append(n)
        if max_frame and attrs['frame'] > max_frame:
            del_nodes.append(n)
    g_out = g.copy()
    g_out.remove_nodes_from(set(del_nodes))
    return g_out


def plot_graph(g, ax, color=BLUE):
    """Draw a lineage tree in the reference style: single-color nodes,
    arrow-free edges, no axis spines."""
    pos = {n: (data['x'], data['frame']) for n, data in g.nodes.items()}
    for u, v in g.edges():
        (xu, yu), (xv, yv) = pos[u], pos[v]
        ax.plot([xu, xv], [yu, yv], color=color, lw=EDGE_WIDTH, zorder=1)
    for n, (x, y) in pos.items():
        ax.scatter([x], [y], s=NODE_SIZE, color=color,
                   zorder=2, edgecolors="none")
    ax.axis("off")

In [13]:
#| code-fold: true
#| code-summary: Show code to compute metrics on tracking solutions using [`traccuracy`](https://github.com/live-image-tracking-tools/traccuracy)


from traccuracy.matchers import Matched
from traccuracy import TrackingGraph
from traccuracy.metrics import CTCMetrics, DivisionMetrics, BasicMetrics, CHOTAMetric

def compute_metrics(solution):
    """Score a solution against the full ground-truth tree with traccuracy."""
    matched = Matched(
        TrackingGraph(gt.copy(), frame_key='frame'),
        TrackingGraph(solution, frame_key='frame'),
        mapping=[(nid, nid) for nid in gt.nodes if nid in solution.nodes],
        matcher_info={},
    )
    results = {
        **CTCMetrics().compute(matched).results,
        **DivisionMetrics().compute(matched).results['Frame Buffer 0'],
        **BasicMetrics().compute(matched).results,
        **CHOTAMetric().compute(matched).results
    }
    return results

In [14]:
#| label: fig-sparse-edges
#| fig-cap: |-
#|   Tracking solutions with common failure modes, each scored with a
#|   panel of tracking metrics. Trees show a single representative lineage
#|   truncated to 10 frames; the metric tables are computed over the full
#|   dataset (10 lineages, 30 frames).
#|   Panel b shows a solution where every fourth node (detection) is missing.
#|   Panel c shows a solution where every third edge is missing. Panel d shows
#|   a solution where no divisions were predicted. Division precision/F1 are
#|   undefined (n/a) for the no-division solution because it predicts no divisions.
#| warning: false
#| echo: false

# Generate single trees for plotting
plot_gt = get_single_tree(gt, max_frame=10)
annotate_positions(plot_gt)

plot_no_edge = plot_gt.copy()
sparsify_edges(plot_no_edge)

plot_no_div = plot_gt.copy()
remove_divs(plot_no_div)

plot_no_node = plot_gt.copy()
remove_nodes(plot_no_node)

# Tables report metrics on the full dataset, not the truncated tree drawn above
no_edge_full = gt.copy()
sparsify_edges(no_edge_full)
no_div_full = gt.copy()
remove_divs(no_div_full)
no_node_full = gt.copy()
remove_nodes(no_node_full)
res_no_edge = compute_metrics(no_edge_full)
res_no_div = compute_metrics(no_div_full)
res_no_node = compute_metrics(no_node_full)

# (display label, traccuracy results key) for each row of the metric tables
TABLE_METRICS = [
    ("TRA", "TRA"),
    ("CHOTA", "CHOTA"),
    ("Node F1", "Node F1"),
    ("Edge F1", "Edge F1"),
    ("Div. F1", "Division F1"),
]

def fmt_metric(key, value):
    if value != value:  # NaN, e.g. division precision with no predicted divisions
        return "n/a"
    if key in ("False Negative Edges", "Total GT Edges"):
        return f"{int(value)}"
    return f"{value:.3f}"


def add_metric_table(ax, res):
    ax.axis("off")
    rows = [[label, fmt_metric(key, res[key])] for label, key in TABLE_METRICS]
    tbl = ax.table(cellText=rows, colLabels=["Metric", "Value"],
                   cellLoc="left", colWidths=[0.6, 0.4], loc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(LABEL_FONT_SIZE)
    tbl.scale(1, 1.25)
    for (row, _col), cell in tbl.get_celld().items():
        cell.set_edgecolor("0.8")
        if row == 0:  # header
            cell.set_text_props(fontweight="bold")


# Top row: lineage trees. Bottom row: a metric table under each solution.
# fig = plt.figure(figsize=(7.2, 5.2))
fig = plt.figure(figsize=(5.0, 7.0))
gs = fig.add_gridspec(3, 3, height_ratios=[2.3, 2.3,1.0])
# tree_axes = [fig.add_subplot(gs[0, i]) for i in range(4)]
tree_axes = [fig.add_subplot(gs[0, 1]),fig.add_subplot(gs[1, 0]),fig.add_subplot(gs[1,1]),fig.add_subplot(gs[1,2])]
panels = [
    (plot_gt, "a) Ground Truth", None),
    (plot_no_node, "b) Missing Node\nSolution", res_no_node),
    (plot_no_edge, "c) Sparse Edge\nSolution", res_no_edge),
    (plot_no_div, "d) No Division\nSolution", res_no_div),
]
for axis, (g, title, _res) in zip(tree_axes, panels):
    plot_graph(g, axis)
    axis.set_title(title, fontsize=TITLE_FONT_SIZE, fontweight="bold", pad=6)
    # thin boundary box hugging the panel content
    axis.add_patch(Rectangle(
        (-0.6, -0.7), 4.2, 11.4,
        fill=False, edgecolor="black", linewidth=1.0))
    axis.set_ylim(11.2, -1.2)  # inverted y (frame increases downward)
    axis.set_xlim(-1.0, 4.0)

# Ground truth has no error table; solutions get one each.
for col, (_g, _title, res) in enumerate(panels[1:]):
    table_ax = fig.add_subplot(gs[-1, col])
    if res is None:
        table_ax.axis("off")
    else:
        add_metric_table(table_ax, res)

# tight_layout cannot lay out matplotlib tables, so position panels manually
fig.subplots_adjust(left=0.03, right=0.99, top=0.88, bottom=0.04,
                    wspace=0.05, hspace=0.3)

Evaluating nodes:   0%|          | 0/3790 [00:00<?, ?it/s]

Evaluating nodes: 100%|██████████| 3790/3790 [00:00<00:00, 284062.33it/s]

Evaluating FP edges:   0%|          | 0/2520 [00:00<?, ?it/s]

Evaluating FP edges: 100%|██████████| 2520/2520 [00:00<00:00, 205020.87it/s]

Evaluating FN edges:   0%|          | 0/3780 [00:00<?, ?it/s]

...

Evaluating FP edges:   0%|          | 0/2512 [00:00<?, ?it/s]

Evaluating FP edges: 100%|██████████| 2512/2512 [00:00<00:00, 315395.19it/s]

Evaluating FN edges:   0%|          | 0/3780 [00:00<?, ?it/s]

Evaluating FN edges: 100%|██████████| 3780/3780 [00:00<00:00, 475353.61it/s]

<Figure size 1500x2100 with 7 Axes>